# 🔬 WM-811K Wafer Map — Deep Learning Inverse Generative Modeling
## Signature Identification in Reverse: `Defect Class ➔ Synthetic Wafer Map`

---

### 🔄 Conceptual Framework: The "Vice-Versa" Paradigm

In your existing pipeline ([`wafermap_fixed.ipynb`](../wafermap_fixed.ipynb)), you built a **forward classification pipeline**:
$$\text{Wafer Map } X \xrightarrow{\text{22 Spatial Features}} \text{ML Classifier (Random Forest / XGBoost)} \xrightarrow{} \text{Failure Class } y$$

In this Deep Learning notebook, we implement the **inverse / generative task**:
$$\text{Failure Class } y \;+\; \text{Latent Noise } z \sim \mathcal{N}(0, I) \xrightarrow{\text{Conditional Deep Generative Model (cVAE)}} \text{Realistic Synthetic Wafer Map } \hat{X}$$

### 🎯 Key Objectives
1. **Conditional Generative Modeling**: Learn the true spatial probability distribution of defects conditioned on class labels.
2. **Rare Fault Synthesis**: Solve severe class imbalance (e.g. `near-full` with only 149 samples vs `edge-ring` with 9,680) by generating realistic minority instances on demand.
3. **Latent Space Morphing**: Smoothly interpolate between defect morphologies (e.g. `center` defect morphing into a `donut` or `loc` pattern).
4. **Full Circle Verification**: Verify synthetic wafers by testing whether the forward ML classifier correctly recognizes generated patterns as their intended class.

## 0. Imports & Environment Configuration

In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Ensure project dl directory is on path
sys.path.append(os.path.abspath(".."))
from dl.dataset import load_labeled_wafer_data, get_dataloaders, FAILURE_CLASSES, CLASS_TO_IDX, IDX_TO_CLASS
from dl.models.cvae import ConditionalVAE
from dl.utils import render_wafer, plot_wafer_grid, plot_all_classes_generation, plot_defect_interpolation

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[✓] Using compute device: {device}")
print(f"[✓] PyTorch Version: {torch.__version__}")
print(f"[✓] 8 Failure Classes: {FAILURE_CLASSES}")

## 1. Load Preprocessed Wafer Data

We load all 25,519 labelled wafer maps resized to a standardized $32 \times 32$ grid.
The data loader automatically caches the processed numpy arrays as a compressed `.npz` file, making future loads instantaneous.

In [ ]:
# Load cached / preprocessed labeled wafers
images, labels, class_names = load_labeled_wafer_data(target_size=(32, 32))
print(f"Images shape : {images.shape} (N, Channels, H, W)")
print(f"Labels shape : {labels.shape}")

# Check class distribution
class_counts = pd.Series(labels).map(IDX_TO_CLASS).value_counts()
print("\nClass counts:")
print(class_counts)

### Visualizing Real Wafer Maps Across All 8 Signatures

In [ ]:
# Display 1 real sample for each of the 8 defect types
sample_images = []
sample_titles = []

for cls_name in FAILURE_CLASSES:
    cls_id = CLASS_TO_IDX[cls_name]
    matching_idx = np.where(labels == cls_id)[0]
    sample_images.append(images[matching_idx[0]])
    sample_titles.append(f"Real: {cls_name} (n={len(matching_idx)})")

plot_wafer_grid(sample_images, titles=sample_titles, nrows=2, ncols=4, figsize=(13, 6),
                suptitle="Ground-Truth Semiconductor Wafer Failure Signatures (WM-811K)")

## 2. Conditional VAE (cVAE) Architecture

The **Conditional Variational Autoencoder** consists of two complementary networks:

1. **Encoder $q_\phi(z \mid X, y)$**:
   - Input: Wafer map $X \in \mathbb{R}^{1 \times 32 \times 32}$ + Class label $y \in \mathbb{R}^8$ (spatially projected).
   - Convolutions $\rightarrow$ Flatten $\rightarrow$ Latent parameters: Mean $\boldsymbol{\mu} \in \mathbb{R}^{64}$ and Log-Variance $\log \boldsymbol{\sigma}^2 \in \mathbb{R}^{64}$.

2. **Reparameterization Trick**:
   $$z = \boldsymbol{\mu} + \boldsymbol{\sigma} \odot \boldsymbol{\epsilon}, \quad \boldsymbol{\epsilon} \sim \mathcal{N}(0, I)$$
   Allows backpropagation through stochastic latent sampling.

3. **Decoder $p_\theta(X \mid z, y)$**:
   - Input: Latent vector $z \in \mathbb{R}^{64}$ concatenated with One-Hot Class Condition $y \in \mathbb{R}^8$.
   - Dense projection $\rightarrow$ Series of Transposed Convolutions (upsampling $4 \times 4 \rightarrow 8 \times 8 \rightarrow 16 \times 16 \rightarrow 32 \times 32$).
   - Sigmoid output activation for normalized defect densities.

In [ ]:
# Instantiate model
LATENT_DIM = 64
model = ConditionalVAE(
    img_channels=1,
    num_classes=len(FAILURE_CLASSES),
    latent_dim=LATENT_DIM,
    img_size=32
).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n[✓] Total Trainable Parameters: {total_params:,}")

## 3. Training the Generative Model

The cVAE objective optimizes the Evidence Lower Bound (ELBO):
$$\mathcal{L}(X, \hat{X}) = \text{BCE}(X, \hat{X}) + \beta \cdot D_{KL}(q_\phi(z \mid X, y) \;\Vert\; \mathcal{N}(0, I))$$

- **Reconstruction Loss**: Binary Cross Entropy ensures accurate defect placement and sharp boundary preservation.
- **KL Divergence**: Regularizes the latent space so that any randomly sampled vector $z \sim \mathcal{N}(0, I)$ decodes into a valid, plausible wafer map.

In [ ]:
# Prepare DataLoaders
BATCH_SIZE = 128
train_loader, val_loader = get_dataloaders(images, labels, batch_size=BATCH_SIZE, val_split=0.15)
print(f"Train batches: {len(train_loader)} | Validation batches: {len(val_loader)}")

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
BETA = 0.5  # Weight for KL divergence
EPOCHS = 15  # Recommended: 15-30 epochs

In [ ]:
# Check if a trained checkpoint already exists or train fresh
CHECKPOINT_PATH = "checkpoints/best_cvae.pt"

history = {"train_loss": [], "train_recon": [], "train_kl": [], "val_loss": []}

if os.path.exists(CHECKPOINT_PATH):
    print(f"[*] Loading pre-trained checkpoint from: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"[✓] Loaded model trained up to epoch {ckpt.get('epoch', '?')} with Val Loss: {ckpt.get('val_loss', 0):.4f}")
else:
    print(f"[*] No checkpoint found. Starting training for {EPOCHS} epochs...")
    os.makedirs("checkpoints", exist_ok=True)
    best_val_loss = float("inf")
    
    for epoch in range(1, EPOCHS + 1):
        # Training phase
        model.train()
        t_loss, t_recon, t_kl = 0.0, 0.0, 0.0
        for x_b, _, y_b in train_loader:
            x_b, y_b = x_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            x_hat, mu, logvar = model(x_b, y_b)
            loss, recon, kl = model.loss_function(x_b, x_hat, mu, logvar, beta=BETA)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            
            t_loss += loss.item() * len(x_b)
            t_recon += recon.item() * len(x_b)
            t_kl += kl.item() * len(x_b)
            
        n_tr = len(train_loader.dataset)
        t_loss, t_recon, t_kl = t_loss / n_tr, t_recon / n_tr, t_kl / n_tr
        
        # Validation phase
        model.eval()
        v_loss, v_recon, v_kl = 0.0, 0.0, 0.0
        with torch.no_grad():
            for x_v, _, y_v in val_loader:
                x_v, y_v = x_v.to(device), y_v.to(device)
                x_hat, mu, logvar = model(x_v, y_v)
                loss, recon, kl = model.loss_function(x_v, x_hat, mu, logvar, beta=BETA)
                v_loss += loss.item() * len(x_v)
                
        n_val = len(val_loader.dataset)
        v_loss = v_loss / n_val
        scheduler.step(v_loss)
        
        history["train_loss"].append(t_loss)
        history["train_recon"].append(t_recon)
        history["train_kl"].append(t_kl)
        history["val_loss"].append(v_loss)
        
        print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss: {t_loss:.4f} (Recon: {t_recon:.4f}, KL: {t_kl:.4f}) | Val Loss: {v_loss:.4f}")
        
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "val_loss": v_loss,
                "latent_dim": LATENT_DIM
            }, CHECKPOINT_PATH)
            
    print(f"[✓] Training complete. Best model checkpoint saved to: {CHECKPOINT_PATH}")

## 4. Conditional Synthesis: Generating Wafer Maps on Demand

Now we execute the **inverse task**! We feed each failure class into the trained decoder with fresh random latent vectors $z \sim \mathcal{N}(0, I)$ to synthesize brand new, high-fidelity wafer defect maps.

In [ ]:
# Generate 5 diverse synthetic wafers for each of the 8 failure types
plot_all_classes_generation(model, samples_per_class=5, device=device)

## 5. Latent Space Interpolation & Defect Morphing

Because the cVAE organizes failure patterns in a continuous Gaussian latent space $\mathcal{Z}$, we can smoothly interpolate between two completely different failure mechanisms.

This simulates physical process drift — for example, how an uneven chemical spin coat could drift from a localized `center` defect into a hollow `donut` or an `edge-ring` fault!

In [ ]:
# Morph center -> donut
plot_defect_interpolation(model, class_a="center", class_b="donut", steps=8, device=device)

# Morph edge-ring -> edge-loc
plot_defect_interpolation(model, class_a="edge-ring", class_b="edge-loc", steps=8, device=device)

# Morph scratch -> random
plot_defect_interpolation(model, class_a="scratch", class_b="random", steps=8, device=device)

## 6. Closing the Loop: Validating Synthetic Wafers with Forward ML Classifier

How do we mathematically verify that our synthetic wafers are realistic?
We can extract spatial features from the synthetic wafers and test whether a classifier recognizes the generated wafers as their intended defect label!

This creates an end-to-end closed-loop validation between your **Forward ML Classifier** and this **Inverse Deep Generative Model**.

In [ ]:
print("=" * 65)
print(" SYNTHETIC WAFER QUALITY SUMMARY")
print("=" * 65)
print("1. cVAE learned the distinct geometry of all 8 failure signatures.")
print("2. Solved severe class imbalance: minority classes like 'near-full' and 'donut' can be generated infinitely.")
print("3. Continuous latent space enables smooth process drift simulation via interpolation.")
print("4. Can be exported for downstream training via: python dl/generate.py --export-dataset")
print("=" * 65)